<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Finding How The Data Is Distributed**


Estimated time needed: **30** minutes


In this lab, you will work with a cleaned dataset to perform Exploratory Data Analysis (EDA). You will examine the structure of the data, visualize key variables, and analyze trends related to developer experience, tools, job satisfaction, and other important aspects.


## Objectives


### Install the required libraries


In [ ]:
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn


### Step 1: Import Libraries and Load Data


- Import the `pandas`, `matplotlib.pyplot`, and `seaborn` libraries.


- You will begin with loading the dataset. You can use the pyfetch method if working on JupyterLite. Otherwise, you can use pandas' read_csv() function directly on their local machines or cloud environments.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv('/Users/michalbando/Downloads/Różne/IT/Python/data/survey_data.csv')
df.head()


### Step 2: Examine the Structure of the Data


- Display the column names, data types, and summary information to understand the data structure.

- Objective: Gain insights into the dataset's shape and available variables.


In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
print(f'Ilosc Wierszy: {df.shape[0]}, Ilosc Kolumn: {df.shape[1]}')

df.describe(include=['object'])

### Step 3: Handle Missing Data


- Identify missing values in the dataset.

- Impute or remove missing values as necessary to ensure data completeness.



In [ ]:
df.isna().sum()

In [ ]:
df['JobSat'].value_counts().sort_values(ascending=False)

In [ ]:
protected_cols = ['JobSat', 'YearsCodePro', 'RemoteWork', 'Employment']
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct.sort_values(ascending=False).head(20))

In [ ]:
for name, value in missing_pct.items():
    if value >= 30:
        print(f"{name} , {round(value, 2)}")

cols_to_drop = [col for col in missing_pct[missing_pct > 30].index if col not in protected_cols]

df.drop(columns=cols_to_drop, inplace=True)
print('*'*50)
print(f"Usunięto {len(cols_to_drop)} kolumn z powodu zbyt dużej liczby braków.")

Data Cleaning Decision:

Columns with >30% missing values were removed to maintain data integrity, as they lacked sufficient information for reliable analysis.

Critical variables like JobSat and RemoteWork were retained despite missing values, as they are essential for the project's objectives. These will be handled using "Unknown" labels or by excluding nulls during statistical calculations.

### Step 4: Analyze Key Columns


In [ ]:
df.info(verbose=True)

- Examine key columns such as `Employment`, `JobSat` (Job Satisfaction), and `YearsCodePro` (Professional Coding Experience).

- **Instruction**: Calculate the value counts for each column to understand the distribution of responses.



In [ ]:
df['JobSat']
df['YearsCodePro'] = df['YearsCodePro'].replace('Less than 1 year', '0').replace('More than 50 years', '51').astype(float)

### Step 5: Visualize Job Satisfaction (Focus on JobSat)


- Create a pie chart or KDE plot to visualize the distribution of `JobSat`.

- Provide an interpretation of the plot, highlighting key trends in job satisfaction.


In [ ]:
missing_pct = (df['JobSat'].isnull().sum() / len(df)) * 100

In [ ]:
plt.figure(figsize=(12, 6))

ax = sns.kdeplot(data=df, x='JobSat', fill=True, cut=0, bw_adjust=1.5, hue='RemoteWork')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), title='Work Model')

plt.title(f'Density of Job Satisfaction (Missing: {missing_pct:.1f}%)')
plt.show()

### Interpretation of the plot
The density plot reveals a multimodal distribution with the highest data concentration (peaks) at scores of 8 and 10, reflecting a high overall level of satisfaction within the surveyed group. Comparative analysis suggests that the remote work model features a higher concentration of positive responses in the upper range of the scale (8-10) compared to the in-person model. The left-skewed shape of the curves confirms that extremely dissatisfied individuals represent a minority, while the dominant trend among IT specialists is satisfaction at a level above 7 points.

### Step 6: Programming Languages Analysis


- Compare the frequency of programming languages in `LanguageHaveWorkedWith` and `LanguageWantToWorkWith`.
  
- Visualize the overlap or differences using a Venn diagram or a grouped bar chart.


In [ ]:
worked_counts = df['LanguageHaveWorkedWith'].str.split(';').explode().value_counts()
want_counts = df['LanguageWantToWorkWith'].str.split(';').explode().value_counts()

lang_df = pd.concat([worked_counts, want_counts], axis=1)
lang_df.columns = ['Worked With', 'Want to Work With']

top_langs = lang_df.sort_values(by='Worked With', ascending=False).head(15)

ax = top_langs.plot(kind='bar', figsize=(14, 7), width=0.8)

plt.title('Top 15 Języków: Obecnie używane vs. Pożądane', fontsize=15)
plt.xlabel('Język programowania', fontsize=12)
plt.ylabel('Liczba respondentów', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Kategoria')

plt.tight_layout()
plt.show()

### Step 7: Analyze Remote Work Trends


- Visualize the distribution of RemoteWork by region using a grouped bar chart or heatmap.


In [ ]:
df_plot = df[["Country", "RemoteWork"]].copy()
df_plot['Country'] = df_plot['Country'].fillna('Unknown')
df_plot['RemoteWork'] = df_plot['RemoteWork'].fillna('Unknown')
ct = pd.crosstab(df_plot['Country'], df_plot['RemoteWork'])
top_countries = ct.sum(axis=1).sort_values(ascending=False).head(10).index
ct_top = ct.loc[top_countries]
plt.figure(figsize=(14,7))
ax = ct_top.plot(kind='bar', figsize=(14, 7), width=0.8)
plt.title('Remote Work Distribution by Country (Top 10 Countries)')
plt.xlabel('Country')
plt.ylabel('Number of respondents')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Remote Work', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Step 8: Correlation between Job Satisfaction and Experience


- Analyze the correlation between overall job satisfaction (`JobSat`) and `YearsCodePro`.
  
- Calculate the Pearson or Spearman correlation coefficient.


In [ ]:
pearson = df['YearsCodePro'].corr(df['JobSat'])
print(f"Korelacja Pearsona: {pearson:.3f}")

spearman = df['YearsCodePro'].corr(df['JobSat'], method='spearman')
print(f"Korelacja Spearmana: {spearman:.3f}")

In [ ]:
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')
df['JobSat'] = pd.to_numeric(df['JobSat'], errors='coerce')

df_clean = df.dropna(subset=['YearsCodePro', 'JobSat'])

plt.figure(figsize=(12, 6))

sns.regplot(data=df_clean, x='YearsCodePro', y='JobSat',
            scatter_kws={'alpha':0.2, 's':10},
            line_kws={'color':'red'},
            x_jitter=0.3, y_jitter=0.3)

plt.title('Korelacja: Lata doświadczenia vs Satysfakcja z pracy', fontsize=15)
plt.xlabel('Lata pracy zawodowej (YearsCodePro)', fontsize=12)
plt.ylabel('Satysfakcja (0-10)', fontsize=12)

max_years = int(df_clean['YearsCodePro'].max())
plt.xticks(range(0, max_years + 1, 5))

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

correlation = df_clean['YearsCodePro'].corr(df_clean['JobSat'], method='spearman')
print(f"Współczynnik korelacji Spearmana: {correlation:.3f}")

### Step 9: Cross-tabulation Analysis (Employment vs. Education Level)


- Analyze the relationship between employment status (`Employment`) and education level (`EdLevel`).

- **Instruction**: Create a cross-tabulation using `pd.crosstab()` and visualize it with a stacked bar plot if possible.


In [ ]:
df_plot = df[['EdLevel', 'Employment']].copy()

df_plot['EdLevel'] = df_plot['EdLevel'].fillna('Unknown')
df_plot['Employment'] = df_plot['Employment'].fillna('Unknown')


def simplify_employment(status):
    if 'Employed full-time' in status:
        return 'Full-time'
    elif 'Employed part-time' in status:
        return 'Part-time'
    elif 'Independent contractor' in status or 'freelancer' in status:
        return 'Freelance/Contract'
    else:
        return 'Other/Not employed'

df_plot['Employment_Simplified'] = df_plot['Employment'].apply(simplify_employment)

ed_map = {
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 'Bachelor’s',
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 'Master’s',
    'Some college/university study without earning a degree': 'Some College',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'Secondary School',
    'Associate degree (A.A., A.S., etc.)': 'Associate',
    'Other doctoral degree (Ph.D., Ed.D., etc.)': 'Doctoral',
    'Professional degree (JD, MD, etc.)': 'Professional',
    'I never completed any formal education': 'None',
    'Primary/elementary school': 'Primary'
}
df_plot['EdLevel_Short'] = df_plot['EdLevel'].map(ed_map).fillna(df_plot['EdLevel'])

ct_simplified = pd.crosstab(df_plot['EdLevel_Short'], df_plot['Employment_Simplified'])

ct_simplified = ct_simplified.loc[ct_simplified.sum(axis=1).sort_values(ascending=False).index]



ax = ct_simplified.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='viridis')

plt.title('Status zatrudnienia w zależności od poziomu wykształcenia (Uproszczone)', fontsize=14)
plt.xlabel('Poziom wykształcenia', fontsize=12)
plt.ylabel('Liczba osób', fontsize=12)

plt.xticks(rotation=45, ha='right')

plt.legend(title='Status zatrudnienia', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

### Step 10: Export Cleaned Data


- Save the cleaned dataset to a new CSV file for further use or sharing.


In [ ]:
export_df = globals().get('df_clean', df)
export_df = export_df.drop_duplicates()
out_path = 'cleaned_survey_data.csv'
export_df.to_csv(out_path, index=False)
print(f'Saved cleaned data to: {out_path}')
display(export_df.head())

### Summary:


In this lab, you practiced key skills in exploratory data analysis, including:


- Examining the structure and content of the Stack Overflow survey dataset to understand its variables and data types.

- Identifying and addressing missing data to ensure the dataset's quality and completeness.

- Summarizing and visualizing key variables such as job satisfaction, programming languages, and remote work trends.

- Analyzing relationships in the data using techniques like:
    - Comparing programming languages respondents have worked with versus those they want to work with.
      
    - Exploring remote work preferences by region.

- Investigating correlations between professional coding experience and job satisfaction.

- Performing cross-tabulations to analyze relationships between employment status and education levels.


## Authors:
Ayushi Jain


### Other Contributors:
Rav Ahuja
Lakshmi Holla
Malika


Copyright © IBM Corporation. All rights reserved.
